In [19]:
import pandas as pd
import networkx as nx
from itertools import combinations
import numpy as np

In [2]:
days = [1, 30, 60, 90]

graphs = {}
attrs = {}

for d in days:
    edges = pd.read_csv(f"Part_B/connections_day_{d}.csv")
    nodes = pd.read_csv(f"Part_B/properties_day_{d}.csv")
    
    G = nx.Graph()
    G.add_edges_from(zip(edges['node_i'], edges['node_j']))
    
    nx.set_node_attributes(G, nodes.set_index('id').to_dict('index'))
    
    graphs[d] = G
    attrs[d] = nodes

In [9]:
print(graphs[1].nodes())
print(graphs[30].nodes())

[64, 81, 103, 106, 46, 112, 72, 41, 76, 25, 77, 92, 30, 39, 0, 117, 38, 107, 62, 104, 2, 33, 67, 118, 14, 10, 93, 19, 47, 22, 7, 58, 13, 116, 20, 97, 69, 99, 80, 111]
[0, 117, 58, 10, 13, 19, 23, 1, 4, 2, 62, 104, 3, 7, 9, 14, 21, 26, 39, 69, 89, 96, 116, 12, 17, 20, 37, 71, 86, 108, 5, 24, 6, 28, 30, 43, 77, 8, 25, 29, 63, 93, 22, 11, 53, 74, 15, 118, 67, 72, 16, 18, 47, 97, 64, 81, 31, 34, 32, 33, 92, 112, 51, 35, 56, 36, 40, 38, 107, 59, 50, 41, 76, 42, 44, 46, 55, 48, 52, 49, 57, 65, 60, 85, 61, 87, 66, 80, 68, 99, 82, 70, 83, 75, 73, 84, 111, 90, 105, 91, 110, 95, 106, 94, 101, 114, 115, 102, 98, 100, 119, 113, 103]


In [ ]:
def triadic_closures(G_prev, G_next):
    closures = []
    for u in G_prev.nodes():
        neighbors = list(G_prev.neighbors(u))
        for v, w in combinations(neighbors, 2):
            if not G_prev.has_edge(v, w) and G_next.has_edge(v, w):
                closures.append((v, w))
    return closures


triadic_events = {}
for d1, d2 in zip(days[:-1], days[1:]):
    triadic_events[(d1, d2)] = triadic_closures(graphs[d1], graphs[d2])


2935


In [27]:
def smoker_triadic_events(G_prev, events):
    smokers = {n for n,d in G_prev.nodes(data=True) if d['smokes']==1}
    return [e for e in events if e[0] in smokers or e[1] in smokers]


for d1, d2 in zip(days[:-1], days[1:]):
    all_events = triadic_events[(d1, d2)]
    smoker_events = smoker_triadic_events(graphs[d1], all_events)
    ratio = len(smoker_events) / len(all_events)
    print(f"the ratio from {d1} to {d2} based on triadic closure is {ratio}")




the ratio from 1 to 30 based on triadic closure is 0.3333333333333333
the ratio from 30 to 60 based on triadic closure is 0.9297200714711138
the ratio from 60 to 90 based on triadic closure is 0.787052810902896


In [24]:
def smoker_membership_ratio(G):
    same, total = 0, 0
    for u, v in G.edges():
        if G.nodes[u]['smokes'] == 1 or G.nodes[v]['smokes'] == 1:
            total += 1
            if G.nodes[u]['class_number'] == G.nodes[v]['class_number']:
                same += 1
    return same / total if total > 0 else 0


def nonsmoker_membership_ratio(G):
    same, total = 0, 0
    for u, v in G.edges():
        if G.nodes[u]['smokes'] == 0 and G.nodes[v]['smokes'] == 0:
            total += 1
            if G.nodes[u]['class_number'] == G.nodes[v]['class_number']:
                same += 1
    return same / total if total > 0 else 0


In [25]:
for d in days:
    print(
        d,
        smoker_membership_ratio(graphs[d]),
        nonsmoker_membership_ratio(graphs[d])
    )

1 0.2 0.15
30 0.5565217391304348 0.7560975609756098
60 0.3765932792584009 0.6918238993710691
90 0.7230769230769231 0.8909090909090909


In [32]:
def smoker_focal_similarity(G, feature):
    same = 0
    total = 0
    for u, v in G.edges():
        if G.nodes[u]['smokes'] == 1 or G.nodes[v]['smokes'] == 1:
            total += 1
            if G.nodes[u][feature] == G.nodes[v][feature]:
                same += 1
    return same / total if total > 0 else 0


In [34]:
features = ['club', 'plays_football', 'watches_movies', 'studies']

for d in days:
    print(f"Day {d}")
    for f in features:
        print(f"  {f}: {smoker_focal_similarity(graphs[d], f):.3f}")

    

Day 1
  club: 0.800
  plays_football: 0.600
  watches_movies: 0.600
  studies: 0.000
Day 30
  club: 0.691
  plays_football: 0.578
  watches_movies: 0.548
  studies: 0.222
Day 60
  club: 0.597
  plays_football: 0.552
  watches_movies: 0.527
  studies: 0.443
Day 90
  club: 0.508
  plays_football: 0.612
  watches_movies: 0.629
  studies: 0.679


In [14]:
def new_smokers(G_prev, G_curr):
    return [
        n for n in G_curr.nodes()
        if n in G_prev.nodes() and G_prev.nodes[n]['smokes'] == 0 and G_curr.nodes[n]['smokes'] == 1
    ]


def new_smoker_focal_exposure(G_prev, new_smokers, feature):
    exposed = 0
    for n in new_smokers:
        for nbr in G_prev.neighbors(n):
            if (G_prev.nodes[nbr]['smokes'] == 1 and
                G_prev.nodes[n][feature] == G_prev.nodes[nbr][feature]):
                exposed += 1
                break
    return exposed / len(new_smokers) if new_smokers else 0

In [15]:
features = ['club', 'plays_football', 'watches_movies', 'studies']
for d1, d2 in zip(days[:-1], days[1:]):
    ns = new_smokers(graphs[d1], graphs[d2])
    print(f"{d1} → {d2}")
    for f in features:
        ratio = new_smoker_focal_exposure(graphs[d1], ns, f)
        print(f"  {f}: {ratio:.2f}")

1 → 30
  club: 0.33
  plays_football: 0.33
  watches_movies: 0.17
  studies: 0.00
30 → 60
  club: 0.85
  plays_football: 0.62
  watches_movies: 0.77
  studies: 0.31
60 → 90
  club: 1.00
  plays_football: 0.89
  watches_movies: 0.89
  studies: 0.61


In [16]:
def feature_distribution(G, feature):
    smokers = []
    nonsmokers = []

    for n, d in G.nodes(data=True):
        if d['smokes'] == 1:
            smokers.append(d[feature])
        else:
            nonsmokers.append(d[feature])

    return np.mean(smokers), np.mean(nonsmokers)

In [20]:
features = ['club', 'plays_football', 'watches_movies', 'studies']

for d in days:
    print(f"\nDay {d}")
    for f in features:
        s, ns = feature_distribution(graphs[d], f)
        print(f"{f}: smokers={s:.2f}, non-smokers={ns:.2f}")


Day 1
club: smokers=0.00, non-smokers=0.14
plays_football: smokers=0.20, non-smokers=0.31
watches_movies: smokers=0.20, non-smokers=0.40
studies: smokers=2.40, non-smokers=2.54

Day 30
club: smokers=0.17, non-smokers=0.25
plays_football: smokers=0.55, non-smokers=0.35
watches_movies: smokers=0.59, non-smokers=0.44
studies: smokers=2.62, non-smokers=2.49

Day 60
club: smokers=0.21, non-smokers=0.45
plays_football: smokers=0.60, non-smokers=0.49
watches_movies: smokers=0.60, non-smokers=0.61
studies: smokers=1.52, non-smokers=2.56

Day 90
club: smokers=0.31, non-smokers=0.72
plays_football: smokers=0.71, non-smokers=0.59
watches_movies: smokers=0.76, non-smokers=0.65
studies: smokers=1.09, non-smokers=2.04


In [21]:
def avg_feature(G, nodes, feature):
    return np.mean([G.nodes[n][feature] for n in nodes]) if nodes else 0

In [22]:
for d1, d2 in zip(days[:-1], days[1:]):
    ns = new_smokers(graphs[d1], graphs[d2])
    print(f"\n{d1} → {d2}")
    for f in features:
        before = avg_feature(graphs[d1], ns, f)
        after  = avg_feature(graphs[d2], ns, f)
        print(f"{f}: before={before:.2f}, after={after:.2f}")


1 → 30
club: before=0.17, after=0.17
plays_football: before=0.17, after=0.67
watches_movies: before=0.67, after=0.83
studies: before=2.50, after=2.83

30 → 60
club: before=0.15, after=0.31
plays_football: before=0.38, after=0.69
watches_movies: before=0.54, after=0.54
studies: before=2.00, after=1.85

60 → 90
club: before=0.39, after=0.46
plays_football: before=0.64, after=0.89
watches_movies: before=0.79, after=0.79
studies: before=2.46, after=1.11


In [23]:
def smoking_rate(G):
    return np.mean([d['smokes'] for _, d in G.nodes(data=True)])

In [24]:
for d in days:
    print(d, smoking_rate(graphs[d]))

1 0.125
30 0.25663716814159293
60 0.35294117647058826
90 0.603448275862069
